In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import classical_estimates
from fit_pv import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609041457C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [4]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [5]:
folder = '/home/ulyanov/data/solo/phi/2026/data/'
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'

folder_out = '/home/ulyanov/data/solo/phi/2026/blos/'
#folder_out = 'temp/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T014503_V202604260832C_0644090501.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T080003_V202606261130C_0644090503.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T110003_V202607131830C_0644090504.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T140003_V202607131930C_0644090505.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T170003_V202607131930C_0644090506.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T200003_V202607151630C_0644090507.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T230003_V202607131730C_0644090508.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260410T014503_V202604260832C_0644100501.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260410T050003_V202604260932C_0644

In [6]:
for file in files[-1:]:

    data, header = process(file,
           dark_file=dark_file,
           deadpix_file=deadpix_file,
           prefilter_file=prefilter_file,
           #cavity_file=cavity_file,
           flatfield_file=flat_file,
           ghost_file=ghost_file,
           distortion_file=distortion_file,
           _realign=True,
           _find_center=True,
           _demodulate=True,
           _correct_fringes=True,
           _correct_crosstalk=True,
           _mask=True,
            )

    Blos, Vlos = classical_estimates(data, header)

    #file_out = generate_filename(file, prefix='blos')
    #hdul = fits.HDUList([fits.PrimaryHDU(data=Blos.clip(-1e4,1e4).astype(np.float32), header=header)])
    #hdul.writeto(folder_out + '/' + file_out, overwrite=True)

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(data[4,3], vmin=-30, vmax=30)
plt.tight_layout()

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(Blos, 'gray', vmin=-200, vmax=200)
plt.tight_layout()

In [8]:
plt.figure(figsize=(10,10))
plt.imshow(Vlos, 'seismic', vmin=-3000, vmax=3000)
plt.tight_layout()

In [56]:
from fitting import *

data_ = data[:,3].copy()

mask = data[contpos,0] > 1000

for i in range(6):
    temp = data_[i].copy()
    temp[mask] = np.nan
    data_[i] -= polyfit2d(temp, degree=1)


In [57]:
x, y = 700, 100
h = 50

temp = np.nanmean(data_[:,x-h:x+h,y-h:y+h], axis=(-2,-1))

plt.figure(figsize=(10,8))
plt.plot(wv, temp)
plt.tight_layout()

In [58]:
plt.figure(figsize=(10,10))
plt.imshow(data_[-1] * ~mask, vmin=-10, vmax=10)
plt.tight_layout()

In [1]:
q_V = 299792458 / 6173.341
q_B = q_V * 0.231
q_B

11217.92199685713

In [3]:
def ceildiv(a, b):
    return -(a // -b)

ceildiv(2000, 1024)

2